[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/06_expectation_variance_and_moments/exercises.ipynb)

# Exercises — Module 06: Expectation, Variance and Moments

20 fully solved problems in four tiers: L0 Concept Checks (4), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (4). Every numeric answer below is recomputed by the code cell that follows it.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

## L0 — Concept Checks

### Problem L0.1 — Does Linearity Need Independence?

**Statement.** Two dice are rolled but glued together so they always show the same face. Let $X$ and $Y$ be the two faces. Compute $E[X+Y]$ and $\mathrm{Var}(X+Y)$, and compare with the independent case.

**Intuition.** Expectation adds up marginals and never looks at the joint law, so gluing changes nothing; variance is a quadratic form and the gluing turns on the cross term at full strength.

**Solution**

**Step 1 — the mean.** Here $Y = X$ exactly, which is maximal dependence. Linearity is untouched:

$$
E[X + Y] = E[X] + E[Y] = 3.5 + 3.5 = 7 ,
$$

the same value as for two independent dice.

**Step 2 — the single-die variance.** For a fair die $E[X^2] = \frac{1}{6}(1+4+9+16+25+36) = \frac{91}{6}$, so

$$
\mathrm{Var}(X) = \frac{91}{6} - \left(\frac{7}{2}\right)^2 = \frac{182 - 147}{12} = \frac{35}{12} \approx 2.9167 .
$$

**Step 3 — the two cases.** Glued: $X + Y = 2X$, so $\mathrm{Var}(2X) = 4\mathrm{Var}(X) = \frac{35}{3} \approx 11.67$. Independent: $\mathrm{Var}(X) + \mathrm{Var}(Y) = \frac{35}{6} \approx 5.83$ — exactly half. The general formula explains the gap: $\mathrm{Cov}(X, X) = \mathrm{Var}(X)$, so the cross term contributes an extra $2\mathrm{Var}(X)$.

$$
\boxed{E[X+Y] = 7 \text{ either way}; \quad \mathrm{Var}(X+Y) = \tfrac{35}{3} \text{ (glued) vs } \tfrac{35}{6} \text{ (independent)}}
$$

**Key takeaway:** Dependence is invisible to expectation and decisive for variance — which is why portfolio *returns* add trivially and portfolio *risk* does not.

In [2]:
faces = np.arange(1, 7)
p_die = np.full(6, 1 / 6)
EX = float(faces @ p_die)
VarX = float((faces ** 2) @ p_die) - EX ** 2

glued = 2 * faces                       # the glued sum takes values 2,4,...,12
E_glued = float(glued @ p_die)
V_glued = float((glued ** 2) @ p_die) - E_glued ** 2

pairs = np.add.outer(faces, faces).ravel()   # all 36 independent outcomes
E_indep, V_indep = float(pairs.mean()), float(pairs.var())

print(f"E[X] = {EX}   Var(X) = {VarX:.6f} = 35/12 = {35/12:.6f}")
print(f"glued      : E = {E_glued}  Var = {V_glued:.6f}  (35/3 = {35/3:.6f})")
print(f"independent: E = {E_indep}  Var = {V_indep:.6f}  (35/6 = {35/6:.6f})")
assert abs(E_glued - 7) < 1e-12 and abs(E_indep - 7) < 1e-12
assert abs(V_glued - 35 / 3) < 1e-12 and abs(V_indep - 35 / 6) < 1e-12

E[X] = 3.5   Var(X) = 2.916667 = 35/12 = 2.916667
glued      : E = 7.0  Var = 11.666667  (35/3 = 11.666667)
independent: E = 7.0  Var = 5.833333  (35/6 = 5.833333)


### Problem L0.2 — Uncorrelated but Dependent

**Statement.** Let $X \sim \mathcal{N}(0,1)$ and $Y = X^2$. Show $\mathrm{Cov}(X,Y) = 0$ while $X$ and $Y$ are as dependent as two variables can be.

**Intuition.** Covariance sees only the linear part of a relationship, and a parabola has no linear part when the input is symmetric about 0.

**Solution**

**Step 1 — compute the covariance.** Using $E[X] = 0$, $E[X^2] = 1$, $E[X^3] = 0$ (odd moments vanish by symmetry),

$$
\mathrm{Cov}(X, Y) = E[XY] - E[X]E[Y] = E\left[X^3\right] - 0 \cdot 1 = 0 ,
$$

so $\rho(X,Y) = 0$.

**Step 2 — exhibit the dependence.** $Y$ is a *deterministic function* of $X$: knowing $X$ determines $Y$ exactly, and knowing $Y$ pins $X$ to $\pm\sqrt{Y}$. Independence would require $P(X \le 1,\, Y \le 0.25) = P(X \le 1)P(Y \le 0.25)$. The left side is $P(-0.5 \le X \le 0.5) = 0.3829$; the right side is $0.8413 \times 0.3829 = 0.3222$. They differ, so $X$ and $Y$ are not independent.

**Step 3 — reconcile.** Covariance measures the projection of $Y$ onto $\operatorname{span}\{1, X\}$. The relationship $Y = X^2$ is purely quadratic and symmetric, so its best linear approximation is flat and the projection carries no information.

$$
\boxed{\mathrm{Cov}(X, X^2) = E[X^3] = 0 \text{ yet } X^2 \text{ is a function of } X}
$$

**Key takeaway:** Zero correlation rules out linear predictability only; a correlation matrix can look like white noise while the data lie exactly on a parabola.

In [3]:
n_mc = 2_000_000
Xs = rng.standard_normal(n_mc)
Ys = Xs ** 2
cov_emp = float(np.cov(Xs, Ys)[0, 1])
se = float(np.std(Xs * Ys, ddof=1) / np.sqrt(n_mc))
print(f"empirical Cov(X, X^2) = {cov_emp:+.5f}   Monte Carlo s.e. = {se:.5f}")
assert abs(cov_emp) < 4 * se           # indistinguishable from zero

lhs = float(stats.norm.cdf(0.5) - stats.norm.cdf(-0.5))
rhs = float(stats.norm.cdf(1.0)) * lhs
print(f"P(X<=1, X^2<=0.25) = {lhs:.4f}   P(X<=1)P(X^2<=0.25) = {rhs:.4f}   "
      f"difference = {lhs - rhs:.4f}")
assert lhs - rhs > 0.05                # dependence, despite zero covariance

empirical Cov(X, X^2) = +0.00248   Monte Carlo s.e. = 0.00274
P(X<=1, X^2<=0.25) = 0.3829   P(X<=1)P(X^2<=0.25) = 0.3222   difference = 0.0608


### Problem L0.3 — Averaging Rates

**Statement.** Two servers process a job in 2 s and 8 s, chosen with equal probability. Compute the expected time and the expected *rate*, and reconcile the two.

**Intuition.** The reciprocal is convex, so averaging rates and inverting the average time must disagree, and Jensen fixes the direction of the disagreement.

**Solution**

**Step 1 — the two averages.** Let $T$ be the time, $P(T=2) = P(T=8) = 1/2$:

$$
E[T] = \frac{2+8}{2} = 5\ \text{s}, \qquad E\left[\frac{1}{T}\right] = \frac{1}{2}\left(\frac{1}{2}+\frac{1}{8}\right) = \frac{5}{16} = 0.3125\ \text{s}^{-1}.
$$

**Step 2 — compare with the reciprocal of the mean.** $1/E[T] = 1/5 = 0.2\ \text{s}^{-1} \ne 0.3125$. Jensen's inequality predicts the direction: $x \mapsto 1/x$ is convex on $(0,\infty)$, so

$$
E\left[\frac{1}{T}\right] \ge \frac{1}{E[T]}, \qquad 0.3125 \ge 0.2 . \checkmark
$$

**Step 3 — decide which number answers which question.** If you run one job on a random server, the expected *time* is 5 s. If you run jobs for a fixed wall-clock period and count throughput, the relevant average is the rate $0.3125$ jobs/s — equivalent to a harmonic-mean time of $1/0.3125 = 3.2$ s. Reporting "average speed" when the experiment fixes distance rather than time is the classic benchmark-averaging error.

$$
\boxed{E[T] = 5 \text{ s}, \quad E[1/T] = 0.3125\ \text{s}^{-1} \gt 1/E[T] = 0.2\ \text{s}^{-1}}
$$

**Key takeaway:** Choose the average that matches what the experiment holds fixed; Jensen guarantees the mismatch is systematic, not a rounding artifact.

In [4]:
times = np.array([2.0, 8.0])
p_t = np.array([0.5, 0.5])
ET = float(times @ p_t)
E_rate = float((1 / times) @ p_t)
print(f"E[T]        = {ET:.4f} s")
print(f"E[1/T]      = {E_rate:.4f} 1/s     1/E[T] = {1/ET:.4f} 1/s")
print(f"harmonic-mean time = 1/E[1/T] = {1/E_rate:.4f} s")
assert abs(ET - 5.0) < 1e-12 and abs(E_rate - 5 / 16) < 1e-12
assert E_rate > 1 / ET                 # Jensen, strict because 1/x is strictly convex

E[T]        = 5.0000 s
E[1/T]      = 0.3125 1/s     1/E[T] = 0.2000 1/s
harmonic-mean time = 1/E[1/T] = 3.2000 s


### Problem L0.4 — When Does the Mean Fail to Exist?

**Statement.** For $X$ with density $f(x) = \frac{1}{\pi(1+x^2)}$ (standard Cauchy), show $E[X]$ does not exist, and explain why the "obvious" symmetry answer 0 is wrong.

**Intuition.** Existence is a statement about $E\lvert X\rvert$, not about symmetry; when both halves are infinite, symmetry only tells you how you happened to approach infinity.

**Solution**

**Step 1 — test integrability.** Existence requires $E\left[\lvert X \rvert\right] \lt \infty$. Compute it:

$$
E\left[\lvert X \rvert\right] = \frac{2}{\pi}\int_0^{\infty}\frac{x}{1+x^2}dx = \frac{1}{\pi}\left[\ln\left(1+x^2\right)\right]_0^{\infty} = \infty .
$$

**Step 2 — conclude.** Since $E[X^+] = E[X^-] = \infty$, the definition $E[X] = E[X^+] - E[X^-]$ yields $\infty - \infty$, which is undefined — not zero, and not even $\pm\infty$.

**Step 3 — see why symmetry is not enough.** Symmetry gives only a *principal value*: $\lim_{R\to\infty}\int_{-R}^{R}xf(x)\,dx = 0$. That limit depends on the cut-off; taking $\int_{-R}^{2R}$ instead gives $\frac{1}{\pi}\ln 2 \approx 0.2206 \ne 0$. A quantity whose value depends on how you approach infinity is not a well-defined expectation.

**Step 4 — the practical consequence.** For i.i.d. Cauchy data, $\bar{X}_n$ has *exactly* the standard Cauchy law for every $n$ (a stability property), so averaging $10$ or $10^{9}$ observations gives an equally noisy answer. The law of large numbers does not merely converge slowly; it fails.

$$
\boxed{E\left[\lvert X \rvert\right] = \infty \Rightarrow E[X] \text{ undefined}; \quad \bar{X}_n \overset{d}{=} X_1 \text{ for all } n}
$$

**Key takeaway:** Symmetry is not integrability — always check $E\lvert X \rvert$ before invoking a mean, an LLN, or a CLT.

In [5]:
for R in (10.0, 1e2, 1e4, 1e8):
    truncated_abs = np.log(1 + R ** 2) / np.pi          # int_{-R}^{R} |x| f(x) dx
    asym = np.log((1 + 4 * R ** 2) / (1 + R ** 2)) / (2 * np.pi)  # int_{-R}^{2R} x f(x) dx
    print(f"R={R:>10.0e}:  E|X| truncated at R = {truncated_abs:10.4f}   "
          f"principal value on [-R,2R] = {asym:.6f}")
print(f"limit of the asymmetric cut-off = ln(2)/pi = {np.log(2)/np.pi:.6f}  (not 0)")

samples = rng.standard_cauchy((6, 1_000_000))
for n in (10, 1_000, 100_000, 1_000_000):
    means = samples[:, :n].mean(axis=1)
    print(f"  n={n:>9}: six independent sample means = {means}")

R=     1e+01:  E|X| truncated at R =     1.4690   principal value on [-R,2R] = 0.219449
R=     1e+02:  E|X| truncated at R =     2.9318   principal value on [-R,2R] = 0.220624
R=     1e+04:  E|X| truncated at R =     5.8635   principal value on [-R,2R] = 0.220636
R=     1e+08:  E|X| truncated at R =    11.7270   principal value on [-R,2R] = 0.220636
limit of the asymmetric cut-off = ln(2)/pi = 0.220636  (not 0)
  n=       10: six independent sample means = [ 1.1917  0.4877 -4.9385  1.2899  3.4688  0.0099]
  n=     1000: six independent sample means = [  0.0932  -1.187  -15.8821  -0.2877   0.7444  -1.0749]
  n=   100000: six independent sample means = [-0.5535 -0.9664 -0.1198 -0.6542  1.0052 -0.9096]


  n=  1000000: six independent sample means = [-0.8844 -2.6302 -1.0524  1.48   -1.6872 -0.2031]


## L1 — Foundations

### Problem L1.1 — Expectation by Indicator Decomposition

**Statement.** A deck of $n$ distinct cards is shuffled uniformly. Let $X$ count *fixed points* (cards left in their original position) and $Y$ count *adjacent pairs* $(j, j+1)$ that are still adjacent and in that order. Find $E[X]$, $\mathrm{Var}(X)$ and $E[Y]$.

**Intuition.** Both counts are sums of indicators whose joint law is awkward but whose marginals are one-line probabilities, and linearity needs nothing else.

**Solution**

**Step 1 — fixed points.** Write $X = \sum_{i=1}^{n}\mathbf{1}_i$ with $\mathbf{1}_i = \mathbf{1}\{\text{card } i \text{ in position } i\}$. Each position is uniform over the $n$ cards, so $P(\mathbf{1}_i = 1) = 1/n$, and linearity (the indicators are strongly dependent, which is irrelevant) gives

$$
E[X] = \sum_{i=1}^{n}\frac{1}{n} = 1 .
$$

Remarkably this is independent of $n$: a shuffled deck of 52 and one of 5 both expect exactly one card to stay put.

**Step 2 — second moment of $X$.** For $i \ne j$, $P(\mathbf{1}_i\mathbf{1}_j = 1) = \frac{1}{n(n-1)}$, so

$$
E[X^2] = E\left[\sum_i \mathbf{1}_i\right] + \sum_{i \ne j}P(\mathbf{1}_i\mathbf{1}_j=1) = 1 + n(n-1)\cdot\frac{1}{n(n-1)} = 2 ,
$$

hence $\mathrm{Var}(X) = 2 - 1 = 1$ — the Poisson(1) signature that underlies the classical derangement limit $P(X=0) \to e^{-1}$.

**Step 3 — adjacent pairs, carefully.** Let $\mathbf{1}_j = \mathbf{1}\{\text{card } j+1 \text{ immediately follows card } j\}$ for $j = 1,\dots,n-1$. Glue $j$ and $j+1$ into a single block: the favourable arrangements are the $(n-1)!$ permutations of that block together with the other $n-2$ cards, out of $n!$ total, so

$$
P(\mathbf{1}_j = 1) = \frac{(n-1)!}{n!} = \frac{1}{n} .
$$

(The tempting answer $1/(n-1)$ comes from conditioning on card $j$ not occupying the last slot, an event of probability $\frac{n-1}{n}$; multiplying back gives $\frac{n-1}{n}\cdot\frac{1}{n-1} = \frac1n$.) Hence

$$
E[Y] = (n-1)\cdot\frac{1}{n} = \frac{n-1}{n} \lt 1 .
$$

**Step 4 — sanity check at $n=2$.** The two permutations are $(1,2)$ and $(2,1)$, giving $Y = 1$ and $Y = 0$, so $E[Y] = \tfrac12 = \frac{n-1}{n}$. $\checkmark$

$$
\boxed{E[X] = 1, \quad \mathrm{Var}(X) = 1, \quad E[Y] = \frac{n-1}{n} \quad \text{for every } n \ge 2}
$$

**Key takeaway:** Indicator decomposition plus linearity solves counting problems whose exact distributions are combinatorially painful — but the indicator probability must be computed unconditionally.

In [6]:
from itertools import permutations

print(f"{'n':>3} {'E[X]':>10} {'Var(X)':>10} {'E[Y]':>10} {'(n-1)/n':>10}")
for n in range(2, 8):
    perms = np.array(list(permutations(range(1, n + 1))))
    fixed = (perms == np.arange(1, n + 1)).sum(axis=1)
    adjacent = (perms[:, 1:] == perms[:, :-1] + 1).sum(axis=1)
    eX, vX, eY = fixed.mean(), fixed.var(), adjacent.mean()
    print(f"{n:>3} {eX:>10.6f} {vX:>10.6f} {eY:>10.6f} {(n-1)/n:>10.6f}")
    assert abs(eX - 1.0) < 1e-12 and abs(vX - 1.0) < 1e-12
    assert abs(eY - (n - 1) / n) < 1e-12
print("E[Y] = (n-1)/n exactly, never 1 -- the 1/(n-1) shortcut is wrong")

  n       E[X]     Var(X)       E[Y]    (n-1)/n
  2   1.000000   1.000000   0.500000   0.500000
  3   1.000000   1.000000   0.666667   0.666667
  4   1.000000   1.000000   0.750000   0.750000
  5   1.000000   1.000000   0.800000   0.800000
  6   1.000000   1.000000   0.833333   0.833333
  7   1.000000   1.000000   0.857143   0.857143
E[Y] = (n-1)/n exactly, never 1 -- the 1/(n-1) shortcut is wrong


### Problem L1.2 — LOTUS in Both Directions

**Statement.** Let $X \sim \text{Unif}(0,1)$. Compute $E\left[X^3\right]$, $E\left[e^{X}\right]$ and $E\left[-\ln X\right]$; then identify the law of $-\ln X$ and check the last answer against it.

**Intuition.** LOTUS integrates the transformation against the *original* density, so the law of the transformed variable is a bonus, never a prerequisite.

**Solution**

**Step 1 — apply LOTUS with $f_X \equiv 1$ on $(0,1)$.**

$$
E\left[X^3\right] = \int_0^1 x^3dx = \frac{1}{4}, \qquad E\left[e^{X}\right] = \int_0^1 e^x dx = e - 1 \approx 1.71828 ,
$$

$$
E\left[-\ln X\right] = -\int_0^1 \ln x\,dx = -\left[x\ln x - x\right]_0^1 = -(0 - 1 - 0) = 1 ,
$$

using $x\ln x \to 0$ as $x \downarrow 0$.

**Step 2 — identify the transformed law.** Let $Z = -\ln X$. For $z \gt 0$,

$$
P(Z \gt z) = P\left(-\ln X \gt z\right) = P\left(X \lt e^{-z}\right) = e^{-z} ,
$$

so $Z \sim \text{Exponential}(1)$ and $E[Z] = 1$. $\checkmark$ The two routes agree, which is the point of LOTUS: computing $E[g(X)]$ never *required* identifying the law of $g(X)$.

**Step 3 — a free Jensen check.** $E\left[e^X\right] = 1.71828 \gt e^{E[X]} = e^{0.5} = 1.64872$, as convexity of $e^x$ requires.

$$
\boxed{E[X^3] = \tfrac{1}{4}, \quad E\left[e^X\right] = e-1, \quad E[-\ln X] = 1 \text{ with } -\ln X \sim \text{Exponential}(1)}
$$

**Key takeaway:** LOTUS turns any expectation of a transformed variable into one integral against the original density — deriving the transformed law is optional.

In [7]:
U = rng.uniform(0.0, 1.0, size=2_000_000)
mc = {"E[X^3]": U ** 3, "E[e^X]": np.exp(U), "E[-ln X]": -np.log(U)}
exact = {"E[X^3]": 0.25, "E[e^X]": np.e - 1, "E[-ln X]": 1.0}
for name, sample in mc.items():
    est, se = sample.mean(), sample.std(ddof=1) / np.sqrt(len(sample))
    print(f"{name:>10}: exact = {exact[name]:.6f}   Monte Carlo = {est:.6f} +- {se:.6f}")
    assert abs(est - exact[name]) < 5 * se

ks = stats.kstest(-np.log(U), "expon")
print(f"KS test of -ln(U) against Exponential(1): statistic = {ks.statistic:.5f}, "
      f"p = {ks.pvalue:.3f}")
print(f"Jensen: E[e^X] = {np.e - 1:.5f} > e^(E[X]) = {np.exp(0.5):.5f}")
assert ks.pvalue > 0.01 and np.e - 1 > np.exp(0.5)

    E[X^3]: exact = 0.250000   Monte Carlo = 0.250033 +- 0.000200


    E[e^X]: exact = 1.718282   Monte Carlo = 1.718463 +- 0.000348


  E[-ln X]: exact = 1.000000   Monte Carlo = 0.999246 +- 0.000707


KS test of -ln(U) against Exponential(1): statistic = 0.00054, p = 0.596
Jensen: E[e^X] = 1.71828 > e^(E[X]) = 1.64872


### Problem L1.3 — Variance of a Sum With Correlation

**Statement.** Portfolio weights $w_1 = 0.6$, $w_2 = 0.4$ are applied to assets with $\sigma_1 = 0.20$, $\sigma_2 = 0.30$. Compute the portfolio standard deviation for $\rho = -1, 0, 0.5, 1$, and find the minimum-variance weights when $\rho = 0$.

**Intuition.** Risk is a quadratic form, so the cross term is a dial: turn $\rho$ from $+1$ to $-1$ and the same two assets go from adding risk to cancelling it.

**Solution**

**Step 1 — expand the quadratic form.** For $R = w_1R_1 + w_2R_2$,

$$
\mathrm{Var}(R) = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho\sigma_1\sigma_2 = 0.0144 + 0.0144 + 0.0288\,\rho .
$$

**Step 2 — tabulate.**

| $\rho$ | $\mathrm{Var}(R)$ | $\mathrm{sd}(R)$ |
|---|---|---|
| $-1$ | $0.0000$ | $0.0000$ |
| $0$ | $0.0288$ | $0.1697$ |
| $0.5$ | $0.0432$ | $0.2078$ |
| $1$ | $0.0576$ | $0.2400$ |

At $\rho = -1$ these particular weights hedge perfectly ($0.6 \times 0.2 = 0.4 \times 0.3$), giving zero variance — the mathematical core of hedging. At $\rho = 1$ risk is the weighted average $0.24$ and diversification buys nothing.

**Step 3 — minimum-variance weights at $\rho = 0$.** Minimize $w^2\sigma_1^2 + (1-w)^2\sigma_2^2$: differentiating gives $2w\sigma_1^2 - 2(1-w)\sigma_2^2 = 0$, so

$$
w^{\star} = \frac{\sigma_2^2}{\sigma_1^2+\sigma_2^2} = \frac{0.09}{0.13} = 0.6923, \qquad \mathrm{Var}^\star = \frac{\sigma_1^2\sigma_2^2}{\sigma_1^2+\sigma_2^2} = 0.027692, \quad \mathrm{sd}^\star = 0.1664 .
$$

This is inverse-variance weighting: the safer asset gets the larger weight, and the resulting variance is the harmonic-type combination — the same formula as optimal sensor fusion and as precision-weighted averaging in Bayesian updates.

$$
\boxed{\mathrm{sd}(R) = 0.0000,\ 0.1697,\ 0.2078,\ 0.2400 \text{ for } \rho = -1, 0, 0.5, 1; \quad w^\star = \frac{\sigma_2^2}{\sigma_1^2+\sigma_2^2} = 0.6923}
$$

**Key takeaway:** Correlation, not individual volatility, decides whether combining sources reduces risk — and inverse-variance weights are optimal when the sources are uncorrelated.

In [8]:
s1, s2, w1, w2 = 0.20, 0.30, 0.6, 0.4
for rho in (-1.0, 0.0, 0.5, 1.0):
    Sigma = np.array([[s1 ** 2, rho * s1 * s2], [rho * s1 * s2, s2 ** 2]])
    w = np.array([w1, w2])
    var = float(w @ Sigma @ w)
    print(f"rho = {rho:+.1f}:  Var = {var:.6f}   sd = {np.sqrt(var):.4f}")
assert abs(np.array([w1, w2]) @ np.array([[s1**2, -s1*s2], [-s1*s2, s2**2]])
           @ np.array([w1, w2])) < 1e-15

w_star = s2 ** 2 / (s1 ** 2 + s2 ** 2)
var_star = s1 ** 2 * s2 ** 2 / (s1 ** 2 + s2 ** 2)
grid = np.linspace(0, 1, 100_001)
var_grid = grid ** 2 * s1 ** 2 + (1 - grid) ** 2 * s2 ** 2
print(f"w* analytic = {w_star:.6f}   grid search = {grid[var_grid.argmin()]:.6f}")
print(f"Var* = {var_star:.6f}   sd* = {np.sqrt(var_star):.4f}")
assert abs(grid[var_grid.argmin()] - w_star) < 1e-4
assert abs(var_grid.min() - var_star) < 1e-9

rho = -1.0:  Var = 0.000000   sd = 0.0000
rho = +0.0:  Var = 0.028800   sd = 0.1697
rho = +0.5:  Var = 0.043200   sd = 0.2078
rho = +1.0:  Var = 0.057600   sd = 0.2400
w* analytic = 0.692308   grid search = 0.692310
Var* = 0.027692   sd* = 0.1664


### Problem L1.4 — Moments From an MGF

**Statement.** Let $M_X(t) = \left(1 - 2t\right)^{-3/2}$ for $t \lt 1/2$. Identify the distribution and compute the mean, variance and skewness.

**Intuition.** Differentiating $M$ drags along a product rule; differentiating $\ln M$ produces the cumulants directly, and the first three cumulants *are* mean, variance and third central moment.

**Solution**

**Step 1 — identify the law.** Compare with the Gamma MGF $\left(\frac{\lambda}{\lambda-t}\right)^{\alpha} = \left(1 - t/\lambda\right)^{-\alpha}$: matching $\lambda = 1/2$ and $\alpha = 3/2$ identifies $X \sim \text{Gamma}\left(\tfrac{3}{2}, \tfrac{1}{2}\right) = \chi^2_3$.

**Step 2 — take logarithms and differentiate.**

$$
K_X(t) = -\frac{3}{2}\ln(1-2t) \implies K_X'(t) = \frac{3}{1-2t}, \quad K_X''(t) = \frac{6}{(1-2t)^2}, \quad K_X'''(t) = \frac{24}{(1-2t)^3}.
$$

**Step 3 — evaluate at $t=0$.**

$$
\kappa_1 = E[X] = 3, \qquad \kappa_2 = \mathrm{Var}(X) = 6, \qquad \kappa_3 = \mu_3 = 24 ,
$$

matching the general $\chi^2_k$ facts $E = k$, $\mathrm{Var} = 2k$ with $k = 3$.

**Step 4 — standardize.**

$$
\gamma_1 = \frac{\kappa_3}{\kappa_2^{3/2}} = \frac{24}{6^{3/2}} = \frac{24}{14.6969} = 1.6330 = \sqrt{\frac{8}{k}} .
$$

Strongly right-skewed, as expected for a sum of three squares.

$$
\boxed{X \sim \chi^2_3: \quad E[X] = 3, \; \mathrm{Var}(X) = 6, \; \gamma_1 = \sqrt{8/3} \approx 1.6330}
$$

**Key takeaway:** Differentiate the *log* MGF: cumulants come out directly and central moments need no binomial bookkeeping.

In [9]:
import sympy as sp

t = sp.symbols("t", real=True)
K = -sp.Rational(3, 2) * sp.log(1 - 2 * t)
kappas = [sp.simplify(sp.diff(K, t, j).subs(t, 0)) for j in (1, 2, 3)]
gamma1 = float(kappas[2] / kappas[1] ** sp.Rational(3, 2))
print(f"cumulants from K = ln M : {kappas}")
print(f"skewness gamma_1 = {gamma1:.6f}   sqrt(8/3) = {np.sqrt(8/3):.6f}")
assert kappas == [3, 6, 24]

m, v, s, k = stats.chi2.stats(3, moments="mvsk")
print(f"scipy chi2(3): mean={float(m):.6f} var={float(v):.6f} skew={float(s):.6f} "
      f"excess kurtosis={float(k):.6f}")
assert abs(float(m) - 3) < 1e-12 and abs(float(v) - 6) < 1e-12
assert abs(float(s) - gamma1) < 1e-12

cumulants from K = ln M : [3, 6, 24]
skewness gamma_1 = 1.632993   sqrt(8/3) = 1.632993
scipy chi2(3): mean=3.000000 var=6.000000 skew=1.632993 excess kurtosis=4.000000


### Problem L1.5 — Chebyshev vs Chernoff vs the Truth

**Statement.** For $X \sim \mathcal{N}(0,1)$, compare the Chebyshev bound on $P\left(\lvert X \rvert \ge 2\right)$ and $P\left(\lvert X \rvert \ge 3\right)$ with the exact values and with a two-sided Chernoff bound, and explain the gaps.

**Intuition.** Chebyshev must survive the worst law with variance 1, and that law is much heavier-tailed than a Gaussian; Chernoff knows the whole MGF and so recovers the exponential shape — but only after paying a factor of 2 for being two-sided.

**Solution**

**Step 1 — Chebyshev.** With $\sigma = 1$,

$$
P\left(\lvert X \rvert \ge k\right) \le \frac{1}{k^2}: \qquad k = 2 \Rightarrow 0.2500, \qquad k=3 \Rightarrow 0.1111 .
$$

**Step 2 — the exact Gaussian values.**

$$
P\left(\lvert X \rvert \ge 2\right) = 2\left(1 - \Phi(2)\right) = 0.04550, \qquad P\left(\lvert X \rvert \ge 3\right) = 0.00270 .
$$

So Chebyshev is off by a factor of $5.5$ at $k=2$ and $41.2$ at $k=3$: the gap widens without limit because Chebyshev decays polynomially while the Gaussian tail decays like $e^{-k^2/2}$.

**Step 3 — why Chebyshev is loose, and why it cannot be improved.** Chebyshev uses *only* the second moment, so it must hold for the worst distribution with that variance. That worst case is attained: the three-point law $P(X = \pm k) = \frac{1}{2k^2}$, $P(X = 0) = 1 - \frac{1}{k^2}$ has variance 1 and meets the bound exactly.

**Step 4 — Chernoff, kept on the same footing.** With the Gaussian MGF $M_X(t) = e^{t^2/2}$,

$$
P(X \ge k) \le \inf_{t \gt 0}e^{-tk+t^2/2} = e^{-k^2/2} \quad (\text{optimal } t = k),
$$

which is a *one-sided* bound. Comparing like with like, the two-sided version is $P(\lvert X\rvert \ge k) \le 2e^{-k^2/2}$, giving $0.2707$ at $k=2$ and $0.0222$ at $k=3$.

**Step 5 — read the comparison honestly.** At $k=2$ the two-sided Chernoff bound $0.2707$ is *worse* than Chebyshev's $0.2500$: the exponential decay has not yet paid for the factor 2. The two cross at $k \approx 2.075$. At $k=3$ Chernoff is $5.0\times$ tighter than Chebyshev and sits $8.2\times$ above the exact $0.0027$.

$$
\boxed{\text{Chebyshev } 0.2500,\ 0.1111 \;\text{vs exact}\; 0.0455,\ 0.0027; \quad 2e^{-k^2/2} = 0.2707,\ 0.0222}
$$

**Key takeaway:** Distribution-free bounds pay for their generality, and a better *rate* is not a better *bound* until the argument is large enough to pay for the constant.

In [10]:
from scipy.optimize import brentq

rows = []
for k_val in (2.0, 3.0):
    cheb = 1 / k_val ** 2
    exact = float(2 * stats.norm.sf(k_val))
    chern2 = float(2 * np.exp(-k_val ** 2 / 2))
    rows.append((k_val, cheb, chern2, exact))
    print(f"k={k_val:.0f}:  Chebyshev={cheb:.4f}  Chernoff(2-sided)={chern2:.4f}  "
          f"exact={exact:.5f}   Cheb/exact={cheb/exact:.1f}x  "
          f"Chern/exact={chern2/exact:.1f}x  Cheb/Chern={cheb/chern2:.2f}x")

# numerical inf over t of the one-sided Chernoff objective, k = 3
ts = np.linspace(1e-4, 8.0, 400_001)
obj = np.exp(-ts * 3.0 + ts ** 2 / 2)
print(f"one-sided inf_t e^(-3t+t^2/2) = {obj.min():.6f} at t = {ts[obj.argmin()]:.4f} "
      f"(theory: e^(-4.5) = {np.exp(-4.5):.6f} at t = 3)")
assert abs(obj.min() - np.exp(-4.5)) < 1e-6

cross = brentq(lambda k: 2 * np.exp(-k * k / 2) - 1 / k ** 2, 1.5, 3.0)
print(f"two-sided Chernoff beats Chebyshev only for k > {cross:.4f}")
assert rows[0][2] > rows[0][1] and rows[1][2] < rows[1][1]   # worse at k=2, better at k=3

# three-point law attains Chebyshev exactly
for k_val in (2.0, 3.0):
    vals = np.array([-k_val, 0.0, k_val])
    pr = np.array([1 / (2 * k_val ** 2), 1 - 1 / k_val ** 2, 1 / (2 * k_val ** 2)])
    var3 = float(pr @ vals ** 2)
    print(f"  worst-case law at k={k_val:.0f}: variance={var3:.6f}, "
          f"P(|X|>=k)={pr[0]+pr[2]:.6f} = 1/k^2 = {1/k_val**2:.6f}")
    assert abs(var3 - 1) < 1e-12 and abs(pr[0] + pr[2] - 1 / k_val ** 2) < 1e-12

k=2:  Chebyshev=0.2500  Chernoff(2-sided)=0.2707  exact=0.04550   Cheb/exact=5.5x  Chern/exact=5.9x  Cheb/Chern=0.92x
k=3:  Chebyshev=0.1111  Chernoff(2-sided)=0.0222  exact=0.00270   Cheb/exact=41.2x  Chern/exact=8.2x  Cheb/Chern=5.00x
one-sided inf_t e^(-3t+t^2/2) = 0.011109 at t = 3.0000 (theory: e^(-4.5) = 0.011109 at t = 3)
two-sided Chernoff beats Chebyshev only for k > 2.0752
  worst-case law at k=2: variance=1.000000, P(|X|>=k)=0.250000 = 1/k^2 = 0.250000
  worst-case law at k=3: variance=1.000000, P(|X|>=k)=0.111111 = 1/k^2 = 0.111111


### Problem L1.6 — Conditional Expectation as a Predictor

**Statement.** Let $X \sim \text{Unif}(0,1)$ and, given $X = x$, let $Y \sim \text{Unif}(0, x)$. Compute $E[Y]$ and $\mathrm{Var}(Y)$, and verify the law of total variance.

**Intuition.** Both conditional moments are simple functions of $X$, so the tower property and Eve's law reduce a two-dimensional problem to two one-dimensional ones.

**Solution**

**Step 1 — conditional moments.** For $Y \mid X = x \sim \text{Unif}(0,x)$: $E[Y \mid X] = \frac{X}{2}$ and $\mathrm{Var}(Y \mid X) = \frac{X^2}{12}$.

**Step 2 — mean by the tower property.**

$$
E[Y] = E\left[\frac{X}{2}\right] = \frac{1}{2}\cdot\frac{1}{2} = \frac{1}{4}.
$$

**Step 3 — variance by Eve's law.** With $E[X^2] = \frac{1}{3}$ and $\mathrm{Var}(X) = \frac{1}{12}$,

$$
E\left[\mathrm{Var}(Y \mid X)\right] = \frac{E[X^2]}{12} = \frac{1}{36}, \qquad \mathrm{Var}\left(E[Y \mid X]\right) = \mathrm{Var}\left(\frac{X}{2}\right) = \frac{1}{4}\cdot\frac{1}{12} = \frac{1}{48},
$$

$$
\mathrm{Var}(Y) = \frac{1}{36} + \frac{1}{48} = \frac{4}{144} + \frac{3}{144} = \frac{7}{144} \approx 0.04861 .
$$

**Step 4 — verify directly.** The joint density is $f(x,y) = \frac{1}{x}$ on $0 \lt y \lt x \lt 1$, so

$$
E\left[Y^2\right] = \int_0^1\!\!\int_0^x \frac{y^2}{x}\,dy\,dx = \int_0^1\frac{x^2}{3}dx = \frac{1}{9}, \qquad \mathrm{Var}(Y) = \frac{1}{9} - \frac{1}{16} = \frac{16-9}{144} = \frac{7}{144}. \checkmark
$$

**Step 5 — read the split.** Of the total variance, $\frac{1}{36} = 0.02778$ (57.1%) is irreducible noise given $X$, and $\frac{1}{48} = 0.02083$ (42.9%) is explained by $X$. Even a perfect observation of $X$ leaves 57.1% of the variance of $Y$ in place.

$$
\boxed{E[Y] = \tfrac14, \quad \mathrm{Var}(Y) = \tfrac{7}{144} = \underbrace{\tfrac{1}{36}}_{\text{unexplained}} + \underbrace{\tfrac{1}{48}}_{\text{explained}}}
$$

**Key takeaway:** Eve's law partitions variance into what conditioning removes and what it cannot — the quantitative form of "how much can this feature possibly help?".

In [11]:
n_h = 4_000_000
Xh = rng.uniform(0.0, 1.0, size=n_h)
Yh = rng.uniform(0.0, 1.0, size=n_h) * Xh        # Y | X ~ Unif(0, X)

cond_mean, cond_var = Xh / 2, Xh ** 2 / 12
emp = {
    "E[Y]": (Yh.mean(), 1 / 4),
    "Var(Y)": (Yh.var(), 7 / 144),
    "E[Var(Y|X)]": (cond_var.mean(), 1 / 36),
    "Var(E[Y|X])": (cond_mean.var(), 1 / 48),
}
for name, (est, exact_val) in emp.items():
    print(f"{name:>12}: Monte Carlo = {est:.6f}   exact = {exact_val:.6f}   "
          f"|diff| = {abs(est - exact_val):.2e}")
    assert abs(est - exact_val) < 2e-4

lhs, rhs = 7 / 144, 1 / 36 + 1 / 48
print(f"Eve's law: Var(Y) = {lhs:.10f} = 1/36 + 1/48 = {rhs:.10f}   "
      f"residual = {abs(lhs - rhs):.2e}")
print(f"unexplained share = {(1/36)/(7/144):.4f}, explained share = {(1/48)/(7/144):.4f}")
assert abs(lhs - rhs) < 1e-15

        E[Y]: Monte Carlo = 0.249971   exact = 0.250000   |diff| = 2.89e-05
      Var(Y): Monte Carlo = 0.048561   exact = 0.048611   |diff| = 4.99e-05
 E[Var(Y|X)]: Monte Carlo = 0.027780   exact = 0.027778   |diff| = 1.96e-06
 Var(E[Y|X]): Monte Carlo = 0.020835   exact = 0.020833   |diff| = 1.65e-06
Eve's law: Var(Y) = 0.0486111111 = 1/36 + 1/48 = 0.0486111111   residual = 6.94e-18
unexplained share = 0.5714, explained share = 0.4286


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Bias-Variance Decomposition of Prediction Error

**Statement.** For $y = f(x) + \varepsilon$ with $E[\varepsilon] = 0$, $\mathrm{Var}(\varepsilon) = \sigma^2$, and an estimator $\hat{f}$ trained on a random dataset $D$ independent of $\varepsilon$, decompose $E_{D,\varepsilon}\left[\left(y - \hat{f}(x)\right)^2\right]$ at a fixed test point $x$.

**Intuition.** Insert and subtract the *average* fitted prediction: the error splits into noise, a systematic offset, and resampling jitter, and all three cross terms die of a zero mean.

**Solution**

**Step 1 — split the error.** Write $\bar{f}(x) = E_D\left[\hat{f}(x)\right]$ and insert $\pm\bar{f}(x)$:

$$
y - \hat{f}(x) = \underbrace{\varepsilon}_{\text{noise}} + \underbrace{\left(f(x) - \bar{f}(x)\right)}_{\text{bias}} + \underbrace{\left(\bar{f}(x) - \hat{f}(x)\right)}_{\text{variance}} .
$$

**Step 2 — kill the cross terms.** Square and take expectations over both $D$ and $\varepsilon$:

- $\varepsilon$ is independent of $D$ with mean 0, killing both cross terms that involve it;
- $E_D\left[\bar{f}(x) - \hat{f}(x)\right] = 0$ by definition of $\bar f$, and the bias term is a constant, so their product has zero mean.

**Step 3 — collect.**

$$
E\left[\left(y - \hat{f}(x)\right)^2\right] = \underbrace{\sigma^2}_{\text{irreducible}} + \underbrace{\left(f(x)-\bar{f}(x)\right)^2}_{\text{bias}^2} + \underbrace{E_D\left[\left(\hat{f}(x)-\bar{f}(x)\right)^2\right]}_{\text{variance}} .
$$

**Step 4 — read it as a design rule.** A high-capacity model (deep tree, unregularized network) has small bias but large variance across resamples; a heavily regularized model reverses this. Ridge regression with penalty $\lambda$ moves along the trade-off explicitly: $\lambda = 0$ is (for a correctly specified linear model) unbiased with maximal variance, and increasing $\lambda$ adds bias while shrinking variance. Bagging averages $B$ nearly-independent fits, dividing the variance term by roughly $B$ while leaving bias unchanged — which is why bagging helps unstable learners and does nothing for a stable one.

$$
\boxed{\mathrm{MSE}(x) = \sigma^2 + \text{bias}^2(x) + \mathrm{Var}\left(\hat{f}(x)\right)}
$$

**Key takeaway:** Only two of the three terms are controllable; treating bias as a resource rather than a defect is what makes regularization work.

In [12]:
# Ridge regression on a 1-D design, resampled many times, at a fixed test point.
n_train, n_rep, sigma_eps, x0 = 12, 40_000, 0.5, 0.8
f_true = lambda z: 2.0 * z


def ridge_pred(lam):
    """Return the predictions at x0 over n_rep independent training sets."""
    Xtr = rng.uniform(-1, 1, size=(n_rep, n_train))
    ytr = f_true(Xtr) + sigma_eps * rng.standard_normal((n_rep, n_train))
    beta = (Xtr * ytr).sum(axis=1) / ((Xtr ** 2).sum(axis=1) + lam)   # ridge, no intercept
    return beta * x0


for lam in (0.0, 2.0, 10.0):
    preds = ridge_pred(lam)
    bias2 = (f_true(x0) - preds.mean()) ** 2
    var = preds.var()
    y_test = f_true(x0) + sigma_eps * rng.standard_normal(n_rep)
    sq_err = (y_test - preds) ** 2
    mse, mse_se = float(sq_err.mean()), float(sq_err.std(ddof=1) / np.sqrt(n_rep))
    print(f"lambda={lam:>4.1f}:  MSE={mse:.4f} +- {mse_se:.4f}   sigma^2+bias^2+var="
          f"{sigma_eps**2 + bias2 + var:.4f}   (sigma^2={sigma_eps**2:.4f}, "
          f"bias^2={bias2:.4f}, var={var:.4f})")
    assert abs(mse - (sigma_eps ** 2 + bias2 + var)) < 4 * mse_se

lambda= 0.0:  MSE=0.2943 +- 0.0021   sigma^2+bias^2+var=0.2936   (sigma^2=0.2500, bias^2=0.0000, var=0.0436)


lambda= 2.0:  MSE=0.5820 +- 0.0036   sigma^2+bias^2+var=0.5799   (sigma^2=0.2500, bias^2=0.3023, var=0.0276)
lambda=10.0:  MSE=1.5914 +- 0.0062   sigma^2+bias^2+var=1.5804   (sigma^2=0.2500, bias^2=1.3201, var=0.0103)


### Problem L2.2 — Why Minibatch Gradients Work

**Statement.** A minibatch gradient is $\hat{g}_B = \frac{1}{B}\sum_{i \in \mathcal{B}}\nabla\ell_i(\theta)$ with $\mathcal{B}$ sampled uniformly with replacement from $\{1,\dots,n\}$. Show it is unbiased, compute its covariance, and derive the implied learning-rate scaling.

**Intuition.** A minibatch gradient is a sample mean of i.i.d. draws, so linearity gives unbiasedness for free and Theorem 4.3 gives the $1/B$ variance.

**Solution**

**Step 1 — unbiasedness.** Let $g = \frac{1}{n}\sum_{i=1}^{n}\nabla\ell_i(\theta)$ be the full gradient. Each index is uniform on $\{1,\ldots,n\}$, so $E\left[\nabla\ell_{i}\right] = g$ and by linearity

$$
E\left[\hat{g}_B\right] = \frac{1}{B}\sum_{i \in \mathcal{B}}E\left[\nabla\ell_i\right] = g .
$$

This holds for *every* batch size, including $B=1$ — precisely the condition Robbins-Monro requires for stochastic approximation to converge.

**Step 2 — covariance.** With independent draws and per-sample covariance $\Sigma = \frac{1}{n}\sum_i\left(\nabla\ell_i - g\right)\left(\nabla\ell_i-g\right)^{\top}$,

$$
\mathrm{Cov}\left(\hat{g}_B\right) = \frac{\Sigma}{B}, \qquad E\left\lVert \hat{g}_B - g\right\rVert^2 = \frac{\operatorname{tr}\Sigma}{B}.
$$

The gradient noise falls as $1/B$, so its standard deviation falls as $1/\sqrt{B}$: **quadrupling the batch halves the noise** while quadrupling the compute — the fundamental diminishing return of large-batch training.

**Step 3 — learning-rate scaling.** One SGD step injects parameter noise of covariance $\eta^2\Sigma/B$. Taking $k$ steps at $(\eta, B)$ injects $k\eta^2\Sigma/B$; one step at $(k\eta, kB)$ injects $(k\eta)^2\Sigma/(kB) = k\eta^2\Sigma/B$ — identical. Hence the **linear scaling rule**: multiply the learning rate by the same factor as the batch size to preserve the noise level. It breaks down at very large $B$ when the deterministic curvature term, not the noise, becomes the binding constraint, which is what warmup schedules address.

$$
\boxed{E\left[\hat{g}_B\right] = g, \quad \mathrm{Cov}\left(\hat{g}_B\right) = \frac{\Sigma}{B}; \quad \eta \propto B \text{ preserves the injected noise}}
$$

**Key takeaway:** SGD converges because the gradient estimator is exactly unbiased; batch size buys only a $\sqrt{B}$ noise reduction, which is why its benefit saturates.

In [13]:
n_data, d, n_draws = 500, 3, 20_000
per_sample = rng.standard_normal((n_data, d)) @ np.array([[1.5, 0.4, 0.0],
                                                          [0.0, 1.0, 0.3],
                                                          [0.0, 0.0, 0.7]])
g_full = per_sample.mean(axis=0)
centred = per_sample - g_full
Sigma_g = centred.T @ centred / n_data

print(f"{'B':>4} {'||E[g_B]-g||':>14} {'tr Cov measured':>18} {'tr Sigma / B':>14}")
for B in (1, 4, 16, 64):
    idx = rng.integers(0, n_data, size=(n_draws, B))
    batches = per_sample[idx].mean(axis=1)
    bias = np.linalg.norm(batches.mean(axis=0) - g_full)
    tr_meas = float(np.cov(batches, rowvar=False).trace())
    tr_pred = float(np.trace(Sigma_g) / B)
    print(f"{B:>4} {bias:>14.5f} {tr_meas:>18.5f} {tr_pred:>14.5f}")
    assert abs(tr_meas - tr_pred) / tr_pred < 0.05
    assert bias < 0.05

   B   ||E[g_B]-g||    tr Cov measured   tr Sigma / B
   1        0.00890            3.77032        3.75984
   4        0.00635            0.92677        0.93996
  16        0.00293            0.23328        0.23499
  64        0.00183            0.05875        0.05875


### Problem L2.3 — Baselines in REINFORCE Are Control Variates

**Statement.** The policy-gradient estimator is $\hat{g} = R\,\nabla_\theta\ln\pi_\theta(a)$. Show that subtracting any action-independent baseline $b$ leaves the estimator unbiased, and find the variance-minimizing $b$.

**Intuition.** The score has mean zero, so anything multiplied by it and independent of the action contributes nothing to the mean but plenty to the variance.

**Solution**

**Step 1 — the score has zero mean.**

$$
E\left[\nabla_\theta\ln\pi_\theta(a)\right] = \sum_a \pi_\theta(a)\frac{\nabla_\theta\pi_\theta(a)}{\pi_\theta(a)} = \nabla_\theta\sum_a\pi_\theta(a) = \nabla_\theta 1 = 0 .
$$

**Step 2 — unbiasedness for any constant baseline.**

$$
E\left[(R - b)\nabla_\theta\ln\pi_\theta(a)\right] = E\left[R\,\nabla_\theta\ln\pi_\theta(a)\right] - b\cdot 0 = \nabla_\theta E[R] .
$$

The baseline is free of charge in the mean — the defining property of a control variate.

**Step 3 — minimize the variance.** Write $s = \nabla_\theta\ln\pi_\theta(a)$ (one scalar component). Then

$$
\mathrm{Var}\left[(R-b)s\right] = E\left[(R-b)^2s^2\right] - \left(\nabla_\theta E[R]\right)^2 ,
$$

and only the first term depends on $b$. Differentiating and setting to zero,

$$
-2E\left[(R-b)s^2\right] = 0 \implies b^{\star} = \frac{E\left[R\,s^2\right]}{E\left[s^2\right]} ,
$$

a score-squared-weighted average return.

**Step 4 — what is used in practice.** The simpler choice $b = E[R]$ (a learned value function $V(s)$) captures most of the benefit; the residual $R - V(s)$ is the **advantage**, and the variance reduction factor is $1 - \rho^2$ with $\rho$ the correlation between $Rs$ and $s$.

$$
\boxed{b^{\star} = \frac{E\left[R\,s^2\right]}{E\left[s^2\right]}; \quad \text{any action-independent } b \text{ keeps the estimator unbiased since } E[s] = 0}
$$

**Key takeaway:** Actor-critic methods are variance reduction, not a different objective — the critic exists only to lower the variance of an already-unbiased gradient.

In [14]:
# Two-action softmax policy with logits (theta, 0); reward depends on the action.
theta = 0.4
pi = np.array([np.exp(theta), 1.0])
pi /= pi.sum()
rewards = np.array([10.0, 11.0])          # large common offset: a baseline should help
score = np.array([1 - pi[0], -pi[0]])     # d/dtheta log pi(a) for a = 0, 1

true_grad = float(pi @ (rewards * score))
b_star = float((pi @ (rewards * score ** 2)) / (pi @ score ** 2))


def est_stats(b):
    vals = (rewards - b) * score
    mean = float(pi @ vals)
    var = float(pi @ (vals - mean) ** 2)
    return mean, var


print(f"true gradient = {true_grad:.6f}   b* = {b_star:.6f}   E[R] = {float(pi @ rewards):.6f}")
for b in (0.0, float(pi @ rewards), b_star, 20.0):
    mean, var = est_stats(b)
    print(f"  b={b:>8.4f}: E[estimator]={mean:.6f}  Var={var:.6f}")
    assert abs(mean - true_grad) < 1e-12          # unbiased for every b
grid_b = np.linspace(-50, 50, 200_001)
variances = np.array([est_stats(float(b))[1] for b in grid_b[::200]])
print(f"grid minimiser = {grid_b[::200][variances.argmin()]:.4f}  vs  b* = {b_star:.4f}")
assert est_stats(b_star)[1] <= variances.min() + 1e-9

true gradient = -0.240261   b* = 10.598688   E[R] = 10.401312
  b=  0.0000: E[estimator]=-0.240261  Var=26.989013
  b= 10.4013: E[estimator]=-0.240261  Var=0.009360
  b= 10.5987: E[estimator]=-0.240261  Var=0.000000
  b= 20.0000: E[estimator]=-0.240261  Var=21.235368
grid minimiser = 10.6000  vs  b* = 10.5987


### Problem L2.4 — Fluctuation-Dissipation From the Partition Function

**Statement.** In the canonical ensemble, $Z(\beta) = \sum_i e^{-\beta E_i}$ with $\beta = 1/(k_BT)$. Show that $\ln Z$ is a cumulant generating function for the energy, and derive the heat-capacity relation $\mathrm{Var}(E) = k_BT^2C_V$.

**Intuition.** $Z$ is literally an MGF of the energy evaluated at $t = -\beta$, so its log generates cumulants and the second cumulant must be an energy fluctuation.

**Solution**

**Step 1 — recognise the MGF.** The Boltzmann distribution is $p_i = e^{-\beta E_i}/Z$, so $Z(\beta) = \sum_i e^{-\beta E_i}$ is the moment generating function of the energy evaluated at $t = -\beta$ (with respect to the counting measure over states). Consequently $\ln Z$ generates cumulants in $-\beta$.

**Step 2 — first derivative.**

$$
\frac{\partial \ln Z}{\partial\beta} = \frac{1}{Z}\sum_i\left(-E_i\right)e^{-\beta E_i} = -\sum_i E_i p_i = -\langle E\rangle .
$$

**Step 3 — second derivative.**

$$
\frac{\partial^2\ln Z}{\partial\beta^2} = -\frac{\partial\langle E\rangle}{\partial\beta} = \frac{1}{Z}\sum_i E_i^2 e^{-\beta E_i} - \left(\frac{1}{Z}\sum_i E_ie^{-\beta E_i}\right)^2 = \left\langle E^2\right\rangle - \left\langle E\right\rangle^2 = \mathrm{Var}(E) .
$$

**Step 4 — convert to a heat capacity.** With $C_V = \frac{\partial\langle E\rangle}{\partial T}$ and $\frac{\partial\beta}{\partial T} = -\frac{1}{k_BT^2}$, the chain rule gives

$$
C_V = \frac{\partial\langle E\rangle}{\partial\beta}\cdot\frac{\partial\beta}{\partial T} = -\mathrm{Var}(E)\cdot\left(-\frac{1}{k_BT^2}\right) = \frac{\mathrm{Var}(E)}{k_BT^2} .
$$

**Step 5 — read the consequence.** A **fluctuation** (second cumulant) equals a **response** (heat capacity). For $N$ independent subsystems, cumulant additivity gives $\mathrm{Var}(E) \propto N$ while $\langle E\rangle \propto N$, so relative fluctuations scale as $N^{-1/2}$ — the reason macroscopic thermodynamics looks deterministic, and the reason critical points (where $C_V$ diverges) are exactly where fluctuations stop being negligible.

$$
\boxed{\langle E\rangle = -\partial_\beta \ln Z, \quad \mathrm{Var}(E) = \partial_\beta^2\ln Z = k_BT^2C_V}
$$

**Key takeaway:** The partition function is a cumulant generating function; thermodynamic response coefficients are literally moments of the energy distribution.

In [15]:
# Two-level system with energies {0, eps}; set k_B = 1 so that beta = 1/T.
beta_s, T_s, eps = sp.symbols("beta T epsilon", positive=True)
Z = 1 + sp.exp(-beta_s * eps)
lnZ = sp.log(Z)

E_mean = sp.simplify(-sp.diff(lnZ, beta_s))
E_var = sp.simplify(sp.diff(lnZ, beta_s, 2))
E_mean_T = E_mean.subs(beta_s, 1 / T_s)
C_V = sp.simplify(sp.diff(E_mean_T, T_s))
identity = sp.simplify(E_var.subs(beta_s, 1 / T_s) - T_s ** 2 * C_V)
print(f"<E>      = {sp.simplify(E_mean)}")
print(f"Var(E)   = {sp.factor(E_var)}")
print(f"Var(E) - k_B T^2 C_V simplifies to {identity}")
assert identity == 0

# numerical cross-check by direct summation over the two states
eps_v, T_v = 1.7, 0.9
p_states = np.array([1.0, np.exp(-eps_v / T_v)])
p_states /= p_states.sum()
E_states = np.array([0.0, eps_v])
mean_num = float(p_states @ E_states)
var_num = float(p_states @ (E_states - mean_num) ** 2)
h = 1e-6
mean_at = lambda T: float(np.array([0.0, eps_v]) @ (lambda w: w / w.sum())(
    np.array([1.0, np.exp(-eps_v / T)])))
C_V_num = (mean_at(T_v + h) - mean_at(T_v - h)) / (2 * h)
print(f"direct sum: Var(E)={var_num:.8f}   T^2 C_V={T_v**2 * C_V_num:.8f}   "
      f"residual={abs(var_num - T_v**2 * C_V_num):.2e}")
assert abs(var_num - T_v ** 2 * C_V_num) < 1e-6

<E>      = epsilon/(exp(beta*epsilon) + 1)
Var(E)   = epsilon**2/(4*cosh(beta*epsilon/2)**2)
Var(E) - k_B T^2 C_V simplifies to 0
direct sum: Var(E)=0.32978603   T^2 C_V=0.32978603   residual=1.72e-11


### Problem L2.5 — Importance Sampling and When Its Variance Explodes

**Statement.** To estimate $\mu = E_p[h(X)]$ using samples from $q$, the estimator is $\hat\mu = \frac{1}{N}\sum_i h(X_i)w(X_i)$ with $w = p/q$. Show unbiasedness, give the variance, and analyse $p = \mathcal{N}(0,1)$, $q = \mathcal{N}(0,\sigma_q^2)$.

**Intuition.** Reweighting is exact in the mean by construction; the danger is entirely in the second moment, where the ratio $p/q$ is squared and a light-tailed proposal blows up.

**Solution**

**Step 1 — unbiasedness.** Provided $q \gt 0$ wherever $ph \ne 0$,

$$
E_q\left[h(X)\frac{p(X)}{q(X)}\right] = \int h(x)\frac{p(x)}{q(x)}q(x)\,dx = \int h(x)p(x)\,dx = \mu .
$$

**Step 2 — variance.**

$$
\mathrm{Var}\left(\hat\mu\right) = \frac{1}{N}\left(E_q\left[h^2w^2\right] - \mu^2\right) = \frac{1}{N}\left(\int \frac{h^2p^2}{q}\,dx - \mu^2\right) ,
$$

finite only if $\int h^2p^2/q \lt \infty$. The danger is a proposal with *lighter tails* than the target: then $p/q \to \infty$ and the integral diverges even though every individual sample looks unremarkable.

**Step 3 — the Gaussian case.** Take $h \equiv 1$ (so $\mu = 1$), $p = \mathcal{N}(0,1)$, $q = \mathcal{N}(0,\sigma_q^2)$:

$$
\frac{p(x)^2}{q(x)} = \frac{\sigma_q}{\sqrt{2\pi}}\exp\left(-x^2 + \frac{x^2}{2\sigma_q^2}\right) = \frac{\sigma_q}{\sqrt{2\pi}}\exp\left(-c\,x^2\right), \qquad c = 1 - \frac{1}{2\sigma_q^2} .
$$

This is integrable iff $c \gt 0$, i.e. $\sigma_q^2 \gt \frac{1}{2}$, and using $\int e^{-cx^2}dx = \sqrt{\pi/c}$,

$$
E_q\left[w^2\right] = \frac{\sigma_q}{\sqrt{2\pi}}\sqrt{\frac{\pi}{c}} = \frac{\sigma_q^2}{\sqrt{2\sigma_q^2-1}}, \qquad \mathrm{Var}(\hat\mu) = \frac{1}{N}\left(E_q\left[w^2\right] - 1\right).
$$

**Step 4 — sanity checks.** At $\sigma_q = 1$ the second moment is 1 and the variance is 0 (the proposal *is* the target), and that is the minimum. At $\sigma_q = 3$ it is $9/\sqrt{17} \approx 2.1828$ — a modest penalty for an over-wide proposal. As $\sigma_q^2 \downarrow \frac{1}{2}$ it **diverges**: a proposal narrower than $1/\sqrt{2}$ gives an unbiased estimator with *infinite variance*, whose running average appears to settle and then jumps without warning.

$$
\boxed{\mathrm{Var}(\hat\mu) \lt \infty \iff \int \frac{h^2p^2}{q} \lt \infty; \quad \text{Gaussian case requires } \sigma_q^2 \gt \tfrac12, \; E_q[w^2] = \frac{\sigma_q^2}{\sqrt{2\sigma_q^2-1}}}
$$

**Key takeaway:** Always choose proposals with heavier tails than the target; a light-tailed proposal produces an estimator whose apparent convergence is an illusion.

In [16]:
def second_moment_theory(sq):
    return sq ** 2 / np.sqrt(2 * sq ** 2 - 1) if 2 * sq ** 2 - 1 > 0 else np.inf


N_is = 400_000
print(f"{'sigma_q':>8} {'E_q[w^2] theory':>17} {'Monte Carlo':>14} {'estimate of mu=1':>18}")
for sq in (0.6, 0.8, 1.0, 1.5, 3.0):
    Z_is = rng.normal(0.0, sq, size=N_is)
    w = stats.norm.pdf(Z_is) / stats.norm.pdf(Z_is, scale=sq)
    print(f"{sq:>8.2f} {second_moment_theory(sq):>17.4f} {np.mean(w**2):>14.4f} "
          f"{w.mean():>18.5f}")
    if sq >= 0.8:
        assert abs(np.mean(w ** 2) - second_moment_theory(sq)) / second_moment_theory(sq) < 0.05
    assert abs(w.mean() - 1.0) < 0.05            # unbiased regardless

sq_bad = 1 / np.sqrt(2)
print(f"\nat sigma_q^2 = 1/2 exactly (sigma_q={sq_bad:.4f}) the theoretical second moment is "
      f"{second_moment_theory(sq_bad)}")
Z_bad = rng.normal(0.0, 0.6, size=4_000_000)
w_bad = stats.norm.pdf(Z_bad) / stats.norm.pdf(Z_bad, scale=0.6)
print("sigma_q = 0.6: the sample second moment never settles, because it estimates infinity")
for N_sub in (10_000, 100_000, 1_000_000, 4_000_000):
    chunk = w_bad[:N_sub]
    print(f"  N={N_sub:>9}: mean(w)={chunk.mean():.4f}  mean(w^2)={np.mean(chunk**2):>12.1f}  "
          f"largest weight={chunk.max():>10.1f}  top weight is "
          f"{100*chunk.max()**2/np.sum(chunk**2):.1f}% of sum(w^2)")

 sigma_q   E_q[w^2] theory    Monte Carlo   estimate of mu=1
    0.60               inf        74.3969            1.00755
    0.80            1.2095         1.1929            1.00033
    1.00            1.0000         1.0000            1.00000


    1.50            1.2027         1.2021            0.99972
    3.00            2.1828         2.1683            0.99483

at sigma_q^2 = 1/2 exactly (sigma_q=0.7071) the theoretical second moment is inf


sigma_q = 0.6: the sample second moment never settles, because it estimates infinity
  N=    10000: mean(w)=0.9885  mean(w^2)=         2.4  largest weight=      34.4  top weight is 5.0% of sum(w^2)
  N=   100000: mean(w)=0.9996  mean(w^2)=        13.7  largest weight=     961.3  top weight is 67.4% of sum(w^2)
  N=  1000000: mean(w)=0.9985  mean(w^2)=         8.9  largest weight=    1055.2  top weight is 12.6% of sum(w^2)
  N=  4000000: mean(w)=1.0004  mean(w^2)=        11.8  largest weight=    2060.1  top weight is 9.0% of sum(w^2)


### Problem L2.6 — Error Propagation by the Delta Method

**Statement.** A resistance is computed as $R = V/I$ from measurements $V = 10.0 \pm 0.1$ V and $I = 2.00 \pm 0.02$ A, uncorrelated. Estimate $E[R]$ and $\mathrm{sd}(R)$, and state the correction when $V$ and $I$ are correlated.

**Intuition.** Linearize the nonlinear function at the mean; the variance of the linearization is a quadratic form in the covariance matrix, which is exactly Theorem 4.3.

**Solution**

**Step 1 — linearize.** For $g(V,I) = V/I$, expand to first order about $(\mu_V, \mu_I) = (10, 2)$:

$$
\frac{\partial g}{\partial V} = \frac{1}{I} = 0.5, \qquad \frac{\partial g}{\partial I} = -\frac{V}{I^2} = -2.5 .
$$

**Step 2 — propagate the variance.** With independence, $\mathrm{Var}(R) \approx \left(\frac{\partial g}{\partial V}\right)^2\sigma_V^2 + \left(\frac{\partial g}{\partial I}\right)^2\sigma_I^2$:

$$
\mathrm{Var}(R) \approx (0.5)^2(0.1)^2 + (2.5)^2(0.02)^2 = 0.0025 + 0.0025 = 0.0050 ,
$$

$$
\mathrm{sd}(R) \approx 0.0707\ \Omega, \qquad E[R] \approx 5.00\ \Omega .
$$

**Step 3 — the relative-error form.** For a ratio or product this is cleaner and worth memorizing:

$$
\left(\frac{\sigma_R}{R}\right)^2 \approx \left(\frac{\sigma_V}{V}\right)^2 + \left(\frac{\sigma_I}{I}\right)^2 = (0.01)^2 + (0.01)^2 \implies \frac{\sigma_R}{R} = 1.41\% .
$$

**Step 4 — the second-order bias.** $g$ is convex in $I$, so Jensen gives $E[R] \gt \mu_V/\mu_I$; the second-order correction is $E[R] \approx \frac{\mu_V}{\mu_I}\left(1 + \frac{\sigma_I^2}{\mu_I^2}\right) = 5.00 \times 1.0001$ — negligible here, but not for noisy denominators.

**Step 5 — the correlated case.** The general formula adds the cross term:

$$
\mathrm{Var}(R) \approx \sum_{i,j}\frac{\partial g}{\partial x_i}\frac{\partial g}{\partial x_j}\mathrm{Cov}(x_i,x_j) = 0.0050 + 2(0.5)(-2.5)\,\mathrm{Cov}(V,I) .
$$

Positively correlated $V$ and $I$ (a shared supply drift, say) *reduce* the uncertainty in their ratio — common-mode errors cancel, which is exactly why ratiometric measurement is a standard instrumentation technique.

$$
\boxed{E[R] \approx 5.00\ \Omega, \quad \mathrm{sd}(R) \approx 0.0707\ \Omega \; (1.41\%); \quad \text{correlation adds } 2g_Vg_I\mathrm{Cov}(V,I)}
$$

**Key takeaway:** Error propagation is a first-order variance computation; ignoring covariance terms can overstate or understate an uncertainty badly.

In [17]:
mu_V, mu_I, sd_V, sd_I = 10.0, 2.0, 0.1, 0.02
grad = np.array([1 / mu_I, -mu_V / mu_I ** 2])
n_ep = 2_000_000

print(f"{'rho(V,I)':>9} {'delta method sd':>17} {'Monte Carlo sd':>16} {'MC mean':>10}")
for rho_vi in (0.0, 0.6, -0.6):
    Sigma_vi = np.array([[sd_V ** 2, rho_vi * sd_V * sd_I],
                         [rho_vi * sd_V * sd_I, sd_I ** 2]])
    sd_delta = float(np.sqrt(grad @ Sigma_vi @ grad))
    L = np.linalg.cholesky(Sigma_vi)
    draws = np.array([mu_V, mu_I]) + (L @ rng.standard_normal((2, n_ep))).T
    R_mc = draws[:, 0] / draws[:, 1]
    print(f"{rho_vi:>9.1f} {sd_delta:>17.5f} {R_mc.std():>16.5f} {R_mc.mean():>10.5f}")
    assert abs(sd_delta - R_mc.std()) / sd_delta < 0.02

print(f"relative error (uncorrelated) = {np.sqrt((sd_V/mu_V)**2 + (sd_I/mu_I)**2)*100:.2f}%")
print(f"second-order bias factor = {1 + sd_I**2/mu_I**2:.6f}  ->  E[R] ~ "
      f"{mu_V/mu_I * (1 + sd_I**2/mu_I**2):.6f}")

 rho(V,I)   delta method sd   Monte Carlo sd    MC mean
      0.0           0.07071          0.07071    5.00045


      0.6           0.04472          0.04472    5.00017
     -0.6           0.08944          0.08953    5.00079
relative error (uncorrelated) = 1.41%
second-order bias factor = 1.000100  ->  E[R] ~ 5.000500


## L3 — Challenge Proofs

### Problem L3.1 — Conditional Expectation as an $L^2$ Projection

**Statement.** Prove that $E[X \mid Y]$ is the orthogonal projection of $X$ onto $L^2\left(\sigma(Y)\right)$, deduce the Pythagorean identity, and derive the Rao-Blackwell theorem.

**Intuition.** The defining identity of conditional expectation says the residual is orthogonal to every indicator in $\sigma(Y)$ — and indicators span the whole subspace, so it is orthogonal to everything there.

**Solution**

**Step 1 — set up the Hilbert space.** Work in $L^2(\Omega,\mathcal{F},P)$ with inner product $\langle U, V\rangle = E[UV]$. Let $\mathcal{H} = L^2\left(\sigma(Y)\right)$, the closed subspace of square-integrable functions of $Y$.

**Step 2 — the defining property is orthogonality.** By Definition 3.5 of [`first_principles.ipynb`](first_principles.ipynb), $m(Y) = E[X\mid Y]$ satisfies $E\left[X\mathbf{1}_A\right] = E\left[m(Y)\mathbf{1}_A\right]$ for all $A \in \sigma(Y)$, i.e.

$$
\left\langle X - m(Y),\, \mathbf{1}_A \right\rangle = 0 \quad \text{for every } A \in \sigma(Y).
$$

Finite linear combinations of such indicators are the $\sigma(Y)$-measurable simple functions, which are dense in $\mathcal{H}$, so by continuity of the inner product

$$
\left\langle X - m(Y),\, h(Y)\right\rangle = 0 \quad \text{for every } h(Y) \in \mathcal{H}.
$$

That is exactly $X - m(Y) \perp \mathcal{H}$, so $m(Y) = P_{\mathcal{H}}X$. Uniqueness follows from uniqueness of projections onto closed subspaces.

**Step 3 — Pythagoras.** For any $h(Y) \in \mathcal{H}$, decompose $X - h(Y) = \left(X - m(Y)\right) + \left(m(Y)-h(Y)\right)$; the two pieces are orthogonal by Step 2, so

$$
E\left[\left(X - h(Y)\right)^2\right] = E\left[\left(X-m(Y)\right)^2\right] + E\left[\left(m(Y)-h(Y)\right)^2\right] \ge E\left[\left(X-m(Y)\right)^2\right].
$$

This re-proves optimal prediction, and taking $h \equiv E[X]$ recovers the law of total variance.

**Step 4 — Rao-Blackwell.** Let $\hat\theta$ be an estimator with $E[\hat\theta] = \theta$ and let $T$ be a sufficient statistic. Define $\tilde\theta = E\left[\hat\theta \mid T\right]$; it is a genuine *statistic* precisely because sufficiency makes this conditional expectation free of $\theta$. Then

- *unbiased*: $E[\tilde\theta] = E\left[E[\hat\theta\mid T]\right] = \theta$ by the tower property;
- *no worse*: applying Step 3 with $X = \hat\theta$, $Y = T$, $h \equiv \theta$,

$$
\mathrm{MSE}(\tilde\theta) = E\left[\left(\tilde\theta-\theta\right)^2\right] = E\left[\left(\hat\theta-\theta\right)^2\right] - E\left[\left(\hat\theta - \tilde\theta\right)^2\right] \le \mathrm{MSE}(\hat\theta),
$$

with strict improvement unless $\hat\theta$ was already a function of $T$. $\blacksquare$

$$
\boxed{E[X\mid Y] = P_{L^2(\sigma(Y))}X; \quad \mathrm{Var}\left(E[\hat\theta\mid T]\right) \le \mathrm{Var}(\hat\theta)}
$$

**Key takeaway:** Conditioning is projection: it can only shorten the residual, which is why Rao-Blackwellization never hurts and why conditional-expectation estimators dominate in Monte Carlo.

In [18]:
# Rao-Blackwell in the canonical example: X_1,...,X_n iid Bernoulli(p),
# theta_hat = X_1 (unbiased), T = sum(X_i) sufficient, E[X_1 | T] = T/n.
p_true, n_obs, n_rep_rb = 0.3, 10, 400_000
data = (rng.random((n_rep_rb, n_obs)) < p_true).astype(float)
theta_hat = data[:, 0]
T = data.sum(axis=1)
theta_tilde = T / n_obs                       # = E[X_1 | T], by exchangeability

print(f"E[theta_hat]   = {theta_hat.mean():.5f}   Var = {theta_hat.var():.6f}   "
      f"theory p(1-p) = {p_true*(1-p_true):.6f}")
print(f"E[theta_tilde] = {theta_tilde.mean():.5f}   Var = {theta_tilde.var():.6f}   "
      f"theory p(1-p)/n = {p_true*(1-p_true)/n_obs:.6f}")
print(f"variance ratio = {theta_tilde.var()/theta_hat.var():.4f}   (1/n = {1/n_obs:.4f})")
assert theta_tilde.var() < theta_hat.var()
assert abs(theta_tilde.mean() - p_true) < 5e-3 and abs(theta_hat.mean() - p_true) < 5e-3

# Pythagoras: E[(X - h(Y))^2] = E[(X - m(Y))^2] + E[(m(Y) - h(Y))^2]
h_fixed = p_true
lhs = float(((theta_hat - h_fixed) ** 2).mean())
rhs = float(((theta_hat - theta_tilde) ** 2).mean() + ((theta_tilde - h_fixed) ** 2).mean())
print(f"Pythagoras: {lhs:.6f} = {rhs:.6f}   residual = {abs(lhs - rhs):.2e}")
assert abs(lhs - rhs) < 1e-3

E[theta_hat]   = 0.30092   Var = 0.210369   theory p(1-p) = 0.210000
E[theta_tilde] = 0.30023   Var = 0.020935   theory p(1-p)/n = 0.021000
variance ratio = 0.0995   (1/n = 0.1000)
Pythagoras: 0.210370 = 0.210671   residual = 3.01e-04


### Problem L3.2 — Hoeffding's Lemma and the Chernoff-Hoeffding Bound

**Statement.** Prove that a bounded variable $X \in [a,b]$ with $E[X] = 0$ satisfies $E\left[e^{tX}\right] \le e^{t^2(b-a)^2/8}$, and deduce Hoeffding's inequality for averages.

**Intuition.** Convexity replaces the unknown law by the worst two-point law on the endpoints; the resulting explicit function has second derivative at most $1/4$, and Taylor does the rest.

**Solution**

**Step 1 — convexity bound.** For $x \in [a,b]$ write $x$ as a convex combination of the endpoints, $x = \lambda b + (1-\lambda)a$ with $\lambda = \frac{x-a}{b-a}$. Convexity of $e^{tx}$ gives

$$
e^{tx} \le \frac{x-a}{b-a}e^{tb} + \frac{b-x}{b-a}e^{ta}.
$$

Take expectations and use $E[X] = 0$:

$$
E\left[e^{tX}\right] \le \frac{-a}{b-a}e^{tb} + \frac{b}{b-a}e^{ta}.
$$

**Step 2 — reparameterize.** Let $p = \frac{-a}{b-a} \in [0,1]$ (well defined since $a \le 0 \le b$) and $u = t(b-a)$. The right side becomes

$$
\left(1 - p + pe^{u}\right)e^{-pu} = e^{\phi(u)}, \qquad \phi(u) = -pu + \ln\left(1 - p + pe^{u}\right).
$$

**Step 3 — Taylor bound on $\phi$.** Compute

$$
\phi(0) = 0, \qquad \phi'(u) = -p + \frac{pe^u}{1-p+pe^u}, \qquad \phi'(0) = 0 ,
$$

and with $\pi(u) = \frac{pe^u}{1-p+pe^u} \in (0,1)$,

$$
\phi''(u) = \pi(u)\left(1 - \pi(u)\right) \le \frac{1}{4},
$$

since $z(1-z) \le 1/4$ on $[0,1]$. Taylor's theorem with Lagrange remainder gives $\phi(u) \le \frac{u^2}{8}$, hence

$$
E\left[e^{tX}\right] \le e^{t^2(b-a)^2/8}. \qquad \blacksquare
$$

**Step 4 — Hoeffding's inequality.** Let $X_1,\ldots,X_n$ be independent with $X_i \in [a_i,b_i]$ and $S = \sum_i\left(X_i - E[X_i]\right)$. Independence multiplies MGFs, and the lemma bounds each factor:

$$
E\left[e^{tS}\right] \le \exp\left(\frac{t^2}{8}\sum_{i}(b_i-a_i)^2\right).
$$

Chernoff's step at level $n\epsilon$, optimizing the quadratic in $t$ at $t^\star = \frac{4n\epsilon}{\sum_i(b_i-a_i)^2}$, yields

$$
P\left(\bar{X}_n - \mu \ge \epsilon\right) \le \exp\left(-\frac{2n^2\epsilon^2}{\sum_i (b_i-a_i)^2}\right), \quad\text{and for } X_i \in [0,1]: \; \le e^{-2n\epsilon^2}.
$$

**Step 5 — compare sample complexities.** Chebyshev gives $\sigma^2/(n\epsilon^2)$, so confidence $1-\delta$ needs $n = O\left(\frac{1}{\delta\epsilon^2}\right)$ samples; Hoeffding needs only $n = O\left(\frac{\ln(1/\delta)}{\epsilon^2}\right)$ — the difference between $\delta^{-1}$ and $\ln\delta^{-1}$, which is why every PAC-learning bound uses the exponential form.

$$
\boxed{E\left[e^{tX}\right] \le e^{t^2(b-a)^2/8}; \qquad P\left(\lvert \bar{X}_n - \mu \rvert \ge \epsilon\right) \le 2e^{-2n\epsilon^2} \text{ for } X_i \in [0,1]}
$$

**Key takeaway:** Boundedness alone makes a variable sub-Gaussian with proxy variance $(b-a)^2/4$; that lemma is the engine of essentially all finite-sample generalization bounds.

In [19]:
# (i) the lemma itself, on the worst case: centred two-point laws on [a, b].
worst_slack = np.inf
for _ in range(2000):
    a_l, b_l = -rng.uniform(0.1, 3.0), rng.uniform(0.1, 3.0)
    q = -a_l / (b_l - a_l)                       # P(X = b) making E[X] = 0
    for t_l in np.concatenate([np.linspace(-4, -0.1, 20), np.linspace(0.1, 4, 20)]):
        mgf = (1 - q) * np.exp(t_l * a_l) + q * np.exp(t_l * b_l)
        bound = np.exp(t_l ** 2 * (b_l - a_l) ** 2 / 8)
        worst_slack = min(worst_slack, bound - mgf)
print(f"min over 2000 random two-point laws, |t| >= 0.1, of (bound - MGF) = "
      f"{worst_slack:.3e}  (strictly positive)")
assert worst_slack > 0

# (ii) Hoeffding's inequality against the empirical tail, X_i ~ Bernoulli(1/2) on [0,1].
n_h2, reps, eps = 200, 400_000, 0.08
means = (rng.random((reps, n_h2)) < 0.5).mean(axis=1)
emp = float((np.abs(means - 0.5) >= eps).mean())
hoeff = float(2 * np.exp(-2 * n_h2 * eps ** 2))
cheb = float(0.25 / n_h2 / eps ** 2)
print(f"n={n_h2}, eps={eps}:  empirical={emp:.5f}   Hoeffding={hoeff:.5f}   "
      f"Chebyshev={cheb:.5f}")
assert emp <= hoeff and hoeff < cheb

# (iii) sample complexity to reach delta = 1e-3 at eps = 0.05
delta, eps2 = 1e-3, 0.05
print(f"n needed for delta={delta} at eps={eps2}:  Hoeffding "
      f"{np.ceil(np.log(2/delta)/(2*eps2**2)):.0f}   Chebyshev "
      f"{np.ceil(0.25/(delta*eps2**2)):.0f}")

min over 2000 random two-point laws, |t| >= 0.1, of (bound - MGF) = 1.321e-07  (strictly positive)


n=200, eps=0.08:  empirical=0.02345   Hoeffding=0.15461   Chebyshev=0.19531
n needed for delta=0.001 at eps=0.05:  Hoeffding 1521   Chebyshev 100000


### Problem L3.3 — Stein's Lemma and Its Use in Score Matching

**Statement.** Prove Stein's lemma: for $X \sim \mathcal{N}(\mu,\sigma^2)$ and differentiable $g$ with $E\left\lvert g'(X)\right\rvert \lt \infty$,

$$
E\left[(X-\mu)g(X)\right] = \sigma^2E\left[g'(X)\right] ,
$$

and use it to derive the Gaussian moments and the score-matching identity.

**Intuition.** The Gaussian density satisfies $\varphi' = -x\varphi$, so multiplying by $x$ is the same as differentiating the density — and integration by parts moves that derivative onto $g$.

**Solution**

**Step 1 — reduce to the standard case.** Take $\mu = 0$, $\sigma = 1$ without loss of generality; the general case follows by substituting $X = \mu + \sigma Z$.

**Step 2 — the structural identity.** The standard normal density $\varphi(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ satisfies

$$
\varphi'(x) = -x\,\varphi(x) .
$$

**Step 3 — integrate by parts.** With $u = g$ and $dv = x\varphi(x)dx$, so $v = -\varphi(x)$,

$$
E\left[Xg(X)\right] = \int_{-\infty}^{\infty}g(x)\,x\varphi(x)\,dx = \left[-g(x)\varphi(x)\right]_{-\infty}^{\infty} + \int_{-\infty}^{\infty}g'(x)\varphi(x)\,dx = E\left[g'(X)\right],
$$

where the boundary term vanishes because $\varphi$ decays faster than any polynomial while $E\lvert g'\rvert \lt \infty$ keeps $g$ of subexponential growth. $\blacksquare$

**Step 4 — Gaussian moments in one line.** Take $g(x) = x^{n-1}$: $E\left[X^{n}\right] = (n-1)E\left[X^{n-2}\right]$. Starting from $E[X^0] = 1$ and $E[X] = 0$, the recursion gives $E[X^2] = 1$, $E[X^4] = 3$, $E[X^6] = 15$ — the double factorial $(n-1)!!$ without a single integral.

**Step 5 — the covariance form.** For a jointly Gaussian pair the same argument gives $\mathrm{Cov}\left(X, g(Y)\right) = \mathrm{Cov}(X,Y)E\left[g'(Y)\right]$, the identity behind Gaussian-input analyses of neural networks (for instance deriving that a ReLU unit's output correlation is an arcsine kernel).

**Step 6 — score matching.** Stein's lemma generalizes: for any smooth density $p$ with score $s_p = \nabla\ln p$ and vanishing boundary flux,

$$
E_p\left[s_p(X)g(X)\right] = -E_p\left[g'(X)\right].
$$

Applying this to the Fisher-divergence objective $J(\theta) = \frac{1}{2}E_p\left\lVert s_\theta(X) - s_p(X)\right\rVert^2$ eliminates the unknown $s_p$: expanding the square, the cross term $-E_p\left[s_\theta^{\top}s_p\right]$ becomes $E_p\left[\nabla\cdot s_\theta\right]$, giving Hyvärinen's tractable objective

$$
J(\theta) = E_p\left[\operatorname{tr}\left(\nabla s_\theta(X)\right) + \frac{1}{2}\left\lVert s_\theta(X)\right\rVert^2\right] + \text{const}.
$$

This is computable from samples alone, with no normalizing constant — the foundation of score-based generative models and diffusion training.

$$
\boxed{E\left[(X-\mu)g(X)\right] = \sigma^2E\left[g'(X)\right]; \quad E\left[X^n\right] = (n-1)E\left[X^{n-2}\right] \text{ for } X\sim\mathcal N(0,1)}
$$

**Key takeaway:** Integration by parts against a Gaussian trades a moment for a derivative — a trick that yields Gaussian moments instantly and makes unnormalized density estimation tractable.

In [20]:
mu_s, sd_s, n_s = 1.0, 2.0, 4_000_000
Xs2 = rng.normal(mu_s, sd_s, size=n_s)
for name, g_fun, gp_fun in [("g(x)=x^3", lambda z: z ** 3, lambda z: 3 * z ** 2),
                            ("g(x)=sin x", np.sin, np.cos),
                            ("g(x)=1/(1+x^2)", lambda z: 1 / (1 + z ** 2),
                             lambda z: -2 * z / (1 + z ** 2) ** 2)]:
    lhs_s = float(np.mean((Xs2 - mu_s) * g_fun(Xs2)))
    rhs_s = float(sd_s ** 2 * np.mean(gp_fun(Xs2)))
    se_s = float(np.std((Xs2 - mu_s) * g_fun(Xs2), ddof=1) / np.sqrt(n_s))
    print(f"{name:>16}: E[(X-mu)g(X)] = {lhs_s:9.5f}   sigma^2 E[g'(X)] = {rhs_s:9.5f}   "
          f"|diff| = {abs(lhs_s - rhs_s):.4f}  (s.e. {se_s:.4f})")
    assert abs(lhs_s - rhs_s) < 6 * se_s

moments = [1.0, 0.0]
for order in range(2, 9):
    moments.append((order - 1) * moments[order - 2])
exact_moments = [float(stats.norm.moment(order)) for order in range(9)]
print(f"Stein recursion moments : {moments}")
print(f"scipy standard normal   : {exact_moments}")
assert np.allclose(moments, exact_moments)

        g(x)=x^3: E[(X-mu)g(X)] =  59.85836   sigma^2 E[g'(X)] =  59.98932   |diff| = 0.1310  (s.e. 0.0972)


      g(x)=sin x: E[(X-mu)g(X)] =   0.29182   sigma^2 E[g'(X)] =   0.29154   |diff| = 0.0003  (s.e. 0.0007)


  g(x)=1/(1+x^2): E[(X-mu)g(X)] =  -0.27048   sigma^2 E[g'(X)] =  -0.27067   |diff| = 0.0002  (s.e. 0.0003)
Stein recursion moments : [1.0, 0.0, 1.0, 0.0, 3.0, 0.0, 15.0, 0.0, 105.0]
scipy standard normal   : [1.0, 0.0, 1.0, 0.0, 3.0, 0.0, 15.000000000000004, 0.0, 105.00000000000001]


### Problem L3.4 — Moments Do Not Always Determine a Distribution

**Statement.** State a sufficient condition for moment determinacy, prove that a finite MGF near 0 suffices, and contrast it with a heavy-tailed counterexample.

**Intuition.** A finite MGF on an interval makes $t \mapsto E[e^{tX}]$ analytic, and an analytic function is pinned down by its derivatives at one point; without that analyticity the moments are just a sequence of numbers, and many laws can share it.

**Solution**

**Step 1 — the sufficient condition.** If $M_X(t) = E\left[e^{tX}\right] \lt \infty$ for all $t \in (-t_0, t_0)$ with some $t_0 \gt 0$, then the moment sequence $\left(E[X^n]\right)_{n\ge0}$ determines $F_X$ uniquely.

**Step 2 — why.** Finiteness of $M_X$ on a real neighbourhood of 0 lets us extend $z \mapsto E\left[e^{zX}\right]$ to the complex strip $\lvert \operatorname{Re}z \rvert \lt t_0$, where dominated convergence shows it is **analytic**: the difference quotients converge because $\left\lvert e^{zX}\right\rvert \le e^{\lvert \operatorname{Re}z\rvert \lvert X\rvert}$ is integrable. An analytic function is determined by its derivatives at a point, so the moments $E[X^n] = M_X^{(n)}(0)$ determine $M_X$ on the whole strip. Restricting to the imaginary axis recovers the characteristic function $\varphi_X(t) = M_X(it)$, and Lévy's inversion theorem recovers $F_X$ from $\varphi_X$ uniquely. $\blacksquare$

**Step 3 — a weaker condition.** Carleman's condition requires no MGF at all:

$$
\sum_{n=1}^{\infty}\left(E\left[X^{2n}\right]\right)^{-1/(2n)} = \infty ,
$$

which permits moments growing as fast as $n!$ but not as fast as $e^{n^2}$.

**Step 4 — the counterexample.** For $X = e^{Y}$ with $Y \sim \mathcal{N}(0,1)$ (standard Lognormal), the moments are $E\left[X^{n}\right] = e^{n^2/2}$ — finite for every $n$, yet growing so fast that

$$
\left(E\left[X^{2n}\right]\right)^{-1/(2n)} = e^{-n} \quad\text{and}\quad \sum_n e^{-n} \lt \infty ,
$$

so Carleman fails. The MGF fails too: $E\left[e^{tX}\right] = \infty$ for every $t \gt 0$, because $e^{te^{y}}$ outgrows $e^{-y^2/2}$.

**Step 5 — exhibit the ambiguity explicitly.** The family

$$
f_\epsilon(x) = f_{\text{LN}}(x)\left[1 + \epsilon\sin\left(2\pi\ln x\right)\right], \qquad \epsilon \in [-1,1],
$$

consists of distinct densities all sharing the Lognormal's moments. Substituting $x = e^u$ turns $\int_0^\infty x^n f_{\text{LN}}(x)\sin(2\pi\ln x)\,dx$ into $\int_{\mathbb{R}} e^{nu}\varphi(u)\sin(2\pi u)\,du$; completing the square gives $e^{n^2/2}\int\varphi(u-n)\sin(2\pi u)\,du = e^{n^2/2}\int\varphi(v)\sin(2\pi v + 2\pi n)\,dv = 0$, because $2\pi n$ is a whole number of periods and $\varphi(v)\sin(2\pi v)$ is odd.

**Step 6 — practical implications.** (i) Method-of-moments estimation is unidentifiable for heavy-tailed families. (ii) Matching a few empirical moments is no evidence of distributional agreement in the tails, where risk lives. (iii) Convergence of moments does *not* imply convergence in distribution unless the limit is moment-determinate — the standard fix is to argue with characteristic functions, which always exist and always determine the law.

$$
\boxed{M_X \text{ finite near } 0 \Rightarrow \text{moments determine } F_X; \quad \text{Lognormal has } E[X^n] = e^{n^2/2} \text{ and is indeterminate}}
$$

**Key takeaway:** Moments are a lossy summary in general; only enough integrability (an MGF near 0, or Carleman's condition) upgrades them to a complete description.

In [21]:
from scipy.integrate import quad

def lognormal_moment(order):
    """E[X^order] for X = exp(Y), Y ~ N(0,1), by quadrature on the shifted integrand."""
    val, _ = quad(lambda u: np.exp(order * u) * stats.norm.pdf(u),
                  order - 12, order + 12, limit=400)
    return val


print(f"{'n':>3} {'E[X^n] = e^(n^2/2)':>20} {'quadrature':>16} {'Carleman term':>15}")
carleman = 0.0
for n in range(1, 7):
    theory = np.exp(n ** 2 / 2)
    numeric = lognormal_moment(n)
    term = lognormal_moment(2 * n) ** (-1 / (2 * n))
    carleman += term
    print(f"{n:>3} {theory:>20.4f} {numeric:>16.4f} {term:>15.6f}  (= e^-n = "
          f"{np.exp(-n):.6f})")
    assert abs(theory - numeric) / theory < 1e-9
    assert abs(term - np.exp(-n)) < 1e-9
print(f"partial Carleman sum (n<=6) = {carleman:.6f}; the full series sums to "
      f"1/(e-1) = {1/(np.e - 1):.6f} < infinity  ->  the condition fails")

# the sin-perturbation shares every moment with the Lognormal
for n in range(0, 6):
    val, err = quad(lambda u: np.exp(n * u) * stats.norm.pdf(u) * np.sin(2 * np.pi * u),
                    -40, 40, limit=400)
    print(f"  n={n}: int x^n f_LN(x) sin(2 pi ln x) dx = {val:+.3e} "
          f"(quad error estimate {err:.1e})")
    assert abs(val) < 1e-6 * max(1.0, np.exp(n ** 2 / 2))

  n   E[X^n] = e^(n^2/2)       quadrature   Carleman term
  1               1.6487           1.6487        0.367879  (= e^-n = 0.367879)
  2               7.3891           7.3891        0.135335  (= e^-n = 0.135335)
  3              90.0171          90.0171        0.049787  (= e^-n = 0.049787)
  4            2980.9580        2980.9580        0.018316  (= e^-n = 0.018316)
  5          268337.2865      268337.2865        0.006738  (= e^-n = 0.006738)
  6        65659969.1373    65659969.1373        0.002479  (= e^-n = 0.002479)
partial Carleman sum (n<=6) = 0.580534; the full series sums to 1/(e-1) = 0.581977 < infinity  ->  the condition fails
  n=0: int x^n f_LN(x) sin(2 pi ln x) dx = +0.000e+00 (quad error estimate 2.9e-22)
  n=1: int x^n f_LN(x) sin(2 pi ln x) dx = -1.171e-16 (quad error estimate 8.9e-09)
  n=2: int x^n f_LN(x) sin(2 pi ln x) dx = -9.281e-16 (quad error estimate 7.6e-11)
  n=3: int x^n f_LN(x) sin(2 pi ln x) dx = +1.629e-14 (quad error estimate 9.4e-10)
  n=4: int x^